# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools

## Imports & Setup

In [1]:
import re
import json
import logging

## Bonus: Logging Setup

Logs every routing decision and result with timestamps — useful for debugging multi-step pipelines.

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger('SmartAgent')
logger.info('Logging initialized')

## 🛠️ Tool 1: Calculator

In [3]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    try:
        safe_expr = re.sub(r'[^0-9\+\-\*\/\(\)\.\s\%]', '', expression).strip()
        if not safe_expr:
            return "Error: No valid expression found"
        result = eval(safe_expr)
        return str(result)
    except ZeroDivisionError:
        return "Error: Division by zero"
    except Exception as e:
        return f"Error in calculation: {str(e)}"

## 🛠️ Tool 2: Keyword Extractor

In [4]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract meaningful keywords from text, filtering stop words."""
    try:
        stop_words = {
            'from', 'this', 'that', 'with', 'have', 'will', 'your',
            'they', 'been', 'were', 'their', 'what', 'when', 'where',
            'which', 'there', 'these', 'those', 'into', 'than', 'then',
            'also', 'some', 'more', 'about', 'each', 'such', 'much'
        }
        words = re.findall(r'\b[a-zA-Z]+\b', text)
        seen = set()
        keywords = []
        for w in words:
            w_lower = w.lower()
            if len(w) > 4 and w_lower not in stop_words and w_lower not in seen:
                keywords.append(w_lower)
                seen.add(w_lower)
        return keywords[:5]
    except Exception:
        return []

## 🛠️ Bonus Tool 3: Word Counter

An additional tool that counts words, characters, and sentences in any input text.

In [5]:
# 🛠️ BONUS TOOL 3: Word Counter

def word_counter(text: str) -> dict:
    """Count words, characters, and sentences in the provided text."""
    try:
        words = text.split()
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        return {
            "word_count": len(words),
            "char_count": len(text),
            "sentence_count": len(sentences)
        }
    except Exception as e:
        return {"error": f"Could not analyze text: {str(e)}"}

## 🤖 Agent Logic

Improved conditional routing using regex:
- `calculate / compute / solve` → Calculator Tool
- `keywords / extract` → Keyword Extractor Tool
- `count words / analyse` → Word Counter Tool *(Bonus)*
- else → General Response

In [6]:
# 🤖 AGENT FUNCTION

def agent(query: str) -> dict:
    """
    Smart Single-Agent that routes queries to appropriate tools.

    Routing:
      'calculate|compute|solve' → Calculator Tool
      'keywords|extract'        → Keyword Extractor Tool
      'count words|analyse'     → Word Counter Tool (Bonus)
      else                      → General Response
    """
    logger.info(f"Received query: '{query}'")
    query_lower = query.lower()

    try:
        # Route 1: Calculation
        if re.search(r'\b(calculate|compute|solve|eval)\b', query_lower):
            logger.info('→ Routing to: Calculator Tool')

            match = re.search(
                r'(?:calculate|compute|solve|eval)\s*([\d\s\+\-\*\/\(\)\.\%]+)',
                query, re.IGNORECASE
            )
            if match:
                expression = match.group(1).strip()
            else:
                expression = re.sub(
                    r'\b(calculate|compute|solve|eval)\b', '',
                    query, flags=re.IGNORECASE
                ).strip()

            result = calculator(expression)
            logger.info(f'Calculator result: {result}')

            return {
                "type": "calculation",
                "expression": expression,
                "result": result
            }

        # Route 2: Keyword Extraction
        elif re.search(r'\b(keywords?|extract|key\s+words?)\b', query_lower):
            logger.info('→ Routing to: Keyword Extractor Tool')

            match = re.search(r'\b(?:from|in|of)\b\s+(.+)$', query, re.IGNORECASE)
            if match:
                text = match.group(1).strip()
            else:
                text = re.sub(
                    r'\b(extract\s+key\s*words?\s*(from)?|keywords?\s*)\b',
                    '', query, flags=re.IGNORECASE
                ).strip() or query

            keywords = extract_keywords(text)
            logger.info(f'Extracted keywords: {keywords}')

            return {
                "type": "keywords",
                "source_text": text,
                "result": keywords
            }

        # Route 3: Word Count (BONUS Tool)
        elif re.search(r'\b(count\s+words?|word\s+count|analyze|analyse)\b', query_lower):
            logger.info('→ Routing to: Word Counter Tool')

            match = re.search(
                r'\b(?:count\s+words?\s+(?:in|of|from)|analyze|analyse)\b\s*(.+)$',
                query, re.IGNORECASE
            )
            text = match.group(1).strip() if match else query
            stats = word_counter(text)
            logger.info(f'Word count stats: {stats}')

            return {
                "type": "word_count",
                "source_text": text,
                "result": stats
            }

        # Route 4: General Response
        else:
            logger.info('→ Routing to: General Response')
            return {
                "type": "general",
                "query": query,
                "result": (
                    f"I received your query: '{query}'. "
                    "Here's what I can help with:\n"
                    "  Math       → 'Calculate 10 + 5 * 2'\n"
                    "  Keywords   → 'Extract keywords from your text here'\n"
                    "  Word Count → 'Count words in your sentence here'"
                )
            }

    except Exception as e:
        logger.error(f'Agent error: {str(e)}')
        return {
            "type": "error",
            "query": query,
            "result": f"An unexpected error occurred: {str(e)}"
        }

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / word_count / general / error",
  "result": ...
}
```

In [7]:
# 🧪 Test Cases

queries = [
    # Required test cases
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    # Bonus test cases
    "Compute 100 * (3 + 7) / 5",
    "Count words in The quick brown fox jumps over the lazy dog",
    "Calculate 9 * 9 - 4 + 2",
    "Calculate invalid expression !!!",
]

print(' RUNNING TEST CASES')

for q in queries:
    print(f"\nQuery   : {q}")
    response = agent(q)
    print(f"Response: {json.dumps(response, indent=2)}")

 RUNNING TEST CASES

Query   : Calculate 20 + 5
Response: {
  "type": "calculation",
  "expression": "20 + 5",
  "result": "25"
}

Query   : Extract keywords from Artificial Intelligence is transforming industries
Response: {
  "type": "keywords",
  "source_text": "Artificial Intelligence is transforming industries",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ]
}

Query   : What is machine learning?
Response: {
  "type": "general",
  "query": "What is machine learning?",
  "result": "I received your query: 'What is machine learning?'. Here's what I can help with:\n  Math       \u2192 'Calculate 10 + 5 * 2'\n  Keywords   \u2192 'Extract keywords from your text here'\n  Word Count \u2192 'Count words in your sentence here'"
}

Query   : Compute 100 * (3 + 7) / 5
Response: {
  "type": "calculation",
  "expression": "100 * (3 + 7) / 5",
  "result": "200.0"
}

Query   : Count words in The quick brown fox jumps over the lazy dog
Response: {
  "

In [8]:
# Interactive Mode

while True:
    user_input = input('Enter query: ').strip()
    if user_input.lower() == 'exit':
        break
    if not user_input:
        continue
    response = agent(user_input)
    print(f"Response: {json.dumps(response, indent=2)}\n")

Enter query: Calculate 6*21/7+8
Response: {
  "type": "calculation",
  "expression": "6*21/7+8",
  "result": "26.0"
}

Enter query: Extract keywords from India is my motherland and I could make every sacrifice for it.
Response: {
  "type": "keywords",
  "source_text": "India is my motherland and I could make every sacrifice for it.",
  "result": [
    "india",
    "motherland",
    "could",
    "every",
    "sacrifice"
  ]
}

Enter query: Where is Paris?
Response: {
  "type": "general",
  "query": "Where is Paris?",
  "result": "I received your query: 'Where is Paris?'. Here's what I can help with:\n  Math       \u2192 'Calculate 10 + 5 * 2'\n  Keywords   \u2192 'Extract keywords from your text here'\n  Word Count \u2192 'Count words in your sentence here'"
}

Enter query: Count words in : Berlin is the capital of Germany.
Response: {
  "type": "word_count",
  "source_text": ": Berlin is the capital of Germany.",
  "result": {
    "word_count": 7,
    "char_count": 35,
    "sentence_co